[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module6/06-video-processing.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module6/06-video-processing.ipynb)

# Module 6.6 — Video Processing
**Module 6: Computer Vision** | Estimated time: 35 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Read videos with `cv2.VideoCapture` and write them with `cv2.VideoWriter`
- Extract frames from a video and inspect properties (FPS, dimensions, frame count)
- Compute dense optical flow with `cv2.calcOpticalFlowFarneback`
- Track sparse feature points with Lucas-Kanade optical flow
- Implement motion detection using frame differencing
- Use `cv2.TrackerCSRT_create()` for object tracking

In [ ]:
!pip install opencv-python-headless --quiet

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import requests, os, tempfile

print(f'OpenCV {cv2.__version__}')
os.makedirs('/tmp/cv_video', exist_ok=True)

def show_frames(frames, titles=None, cols=4, figsize=(16, 4)):
    n = len(frames)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    for i, (ax, fr) in enumerate(zip(axes, frames)):
        if len(fr.shape) == 3:
            fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        ax.imshow(fr, cmap='gray' if len(fr.shape) == 2 else None)
        ax.set_title(titles[i] if titles else f'Frame {i}', fontsize=8)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

# Download a short public-domain video clip (Big Buck Bunny excerpt)
print('Downloading sample video...')
video_url = 'https://www.sample-videos.com/video321/mp4/240/big_buck_bunny_240p_5mb.mp4'
try:
    r = requests.get(video_url, timeout=30)
    if r.status_code == 200:
        with open('/tmp/cv_video/sample.mp4', 'wb') as f:
            f.write(r.content)
        print(f'Video saved: {len(r.content)//1024} KB')
    else:
        raise Exception(f'HTTP {r.status_code}')
except Exception as e:
    print(f'Download failed ({e}). Generating synthetic video...')
    # Synthetic: moving circle on gradient background
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    vw = cv2.VideoWriter('/tmp/cv_video/sample.mp4', fourcc, 24.0, (320, 240))
    for i in range(120):  # 5 seconds at 24 fps
        frame = np.zeros((240, 320, 3), dtype=np.uint8)
        frame[:] = (20, 30, 50)
        x = int(40 + (i / 120) * 240)
        y = int(120 + 60 * np.sin(i * 0.2))
        cv2.circle(frame, (x, y), 25, (0, 200, 255), -1)
        cv2.rectangle(frame, (10, 10), (100, 50), (200, 50, 50), -1)
        vw.write(frame)
    vw.release()
    print('Synthetic video created (120 frames, 24fps, 320×240)')

## VideoCapture: Reading Video Properties

`cv2.VideoCapture` opens a video file or a camera (index 0, 1, …). You can query properties with `.get(propId)` and read frame by frame with `.read()`.

In [ ]:
cap = cv2.VideoCapture('/tmp/cv_video/sample.mp4')

if not cap.isOpened():
    print('ERROR: Cannot open video.')
else:
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    nframes= int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    codec  = int(cap.get(cv2.CAP_PROP_FOURCC))
    duration = nframes / fps if fps > 0 else 0

    print('=== Video Properties ===')
    print(f'Resolution  : {width}×{height}')
    print(f'FPS         : {fps:.2f}')
    print(f'Total frames: {nframes}')
    print(f'Duration    : {duration:.2f} seconds')
    fourcc_str = ''.join([chr((codec >> 8*i) & 0xFF) for i in range(4)])
    print(f'Codec       : {fourcc_str}')

# Extract frames at regular intervals
def extract_frames(path, n_frames=8):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(1, total // n_frames)
    frames = []
    for i in range(n_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    cap.release()
    return frames

frames = extract_frames('/tmp/cv_video/sample.mp4', n_frames=8)
print(f'\nExtracted {len(frames)} frames')
show_frames(frames, [f'Frame {i}' for i in range(len(frames))],
            cols=4, figsize=(16, 4))

## VideoWriter: Saving Processed Video

`cv2.VideoWriter` writes frames to a video file. You must specify the codec, FPS, and frame size upfront. The size must match every frame you write.

In [ ]:
# Read and process: apply edge detection to each frame, save result
out_path = '/tmp/cv_video/edges_output.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
vw = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

cap = cv2.VideoCapture('/tmp/cv_video/sample.mp4')
frame_idx = 0
processed_frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    # Process: Canny edges overlaid on original
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), 50, 150)
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    output = cv2.addWeighted(frame, 0.7, edges_bgr, 0.3, 0)
    # Frame counter
    cv2.putText(output, f'Frame {frame_idx}', (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    vw.write(output)
    if frame_idx % 20 == 0:
        processed_frames.append(output.copy())
    frame_idx += 1

cap.release()
vw.release()
print(f'Processed {frame_idx} frames')
print(f'Output saved: {out_path}')
if os.path.exists(out_path):
    print(f'File size: {os.path.getsize(out_path)//1024} KB')

# Show a few processed frames
if processed_frames:
    show_frames(processed_frames[:4],
                [f'Processed frame {i*20}' for i in range(4)])

## Dense Optical Flow (Farneback)

**Optical flow** describes the apparent motion of pixels between consecutive frames. **Dense** optical flow computes a motion vector for every pixel.

`cv2.calcOpticalFlowFarneback` implements the Gunnar Farneback algorithm. The output is a 2-channel array (dx, dy) — the displacement of each pixel. We visualise it using the HSV colour space: hue encodes direction, saturation encodes magnitude.

In [ ]:
cap = cv2.VideoCapture('/tmp/cv_video/sample.mp4')
ret, frame1 = cap.read()
if not ret:
    print('Could not read first frame.')
else:
    prev_gray = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

    # Skip to a later frame for more interesting motion
    cap.set(cv2.CAP_PROP_POS_FRAMES, 10)
    ret, frame2 = cap.read()
    curr_gray = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Compute dense optical flow
    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray,
        None,
        pyr_scale=0.5,  # pyramid scale
        levels=3,       # pyramid levels
        winsize=15,     # averaging window
        iterations=3,   # iterations per level
        poly_n=5,       # pixel neighbourhood size
        poly_sigma=1.2, # Gaussian std for polynomial expansion
        flags=0
    )
    print(f'Optical flow shape: {flow.shape}  (H×W×2: dx, dy per pixel)')

    # Convert flow to HSV visualisation
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros((*curr_gray.shape, 3), dtype=np.uint8)
    hsv[..., 1] = 255                                        # full saturation
    hsv[..., 0] = ang * 180 / np.pi / 2                    # hue = direction
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)  # value = magnitude
    flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Frame 1')
    axes[1].imshow(cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Frame 2')
    axes[2].imshow(flow_rgb)
    axes[2].set_title('Dense Optical Flow\n(colour = direction, brightness = magnitude)')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout(); plt.show()

cap.release()

## Lucas-Kanade Sparse Optical Flow

**Sparse** optical flow tracks a selected set of feature points rather than every pixel. `cv2.calcOpticalFlowPyrLK` implements the Lucas-Kanade method with a pyramid (for handling large motions).

Use `cv2.goodFeaturesToTrack` to find good points to track in the first frame.

In [ ]:
cap = cv2.VideoCapture('/tmp/cv_video/sample.mp4')
ret, old_frame = cap.read()
cap.release()

if ret:
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

    # Find good features to track
    p0 = cv2.goodFeaturesToTrack(
        old_gray, maxCorners=100, qualityLevel=0.3, minDistance=7)

    cap = cv2.VideoCapture('/tmp/cv_video/sample.mp4')
    cap.set(cv2.CAP_PROP_POS_FRAMES, 10)
    ret, new_frame = cap.read()
    cap.release()

    new_gray = cv2.cvtColor(new_frame, cv2.COLOR_BGR2GRAY)

    # LK parameters
    lk_params = dict(
        winSize=(15, 15),
        maxLevel=2,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
    )

    # Calculate optical flow
    p1, status, err = cv2.calcOpticalFlowPyrLK(
        old_gray, new_gray, p0, None, **lk_params)

    # Select good points
    good_new = p1[status == 1]
    good_old = p0[status == 1]

    print(f'Tracked {len(good_new)} / {len(p0)} initial points')

    # Draw tracks
    mask   = np.zeros_like(old_frame)
    canvas = new_frame.copy()
    colors = np.random.randint(0, 255, (100, 3), dtype=np.uint8)

    for i, (new_pt, old_pt) in enumerate(zip(good_new, good_old)):
        a, b = new_pt.ravel().astype(int)
        c, d = old_pt.ravel().astype(int)
        color = colors[i % 100].tolist()
        cv2.line(mask, (a, b), (c, d), color, 2)
        cv2.circle(canvas, (a, b), 4, color, -1)

    output = cv2.add(canvas, mask)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(cv2.cvtColor(old_frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Frame 1 — {len(p0)} tracked points')
    axes[1].imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'Lucas-Kanade Sparse Flow — {len(good_new)} tracks')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout(); plt.show()

## Practical: Motion Detection with Frame Differencing

The simplest motion detection: compare consecutive frames, threshold the difference, and flag regions that changed significantly.

In [ ]:
def detect_motion(video_path, threshold=25, min_area=500, sample_every=5):
    """
    Simple motion detector using frame differencing.
    Returns a list of (frame_idx, motion_detected, annotated_frame).
    """
    cap = cv2.VideoCapture(video_path)
    results = []
    prev_gray = None
    idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % sample_every == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            gray = cv2.GaussianBlur(gray, (21, 21), 0)

            if prev_gray is not None:
                # Absolute difference between frames
                diff  = cv2.absdiff(prev_gray, gray)
                _, mask = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)

                # Dilate to fill gaps
                kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
                mask   = cv2.dilate(mask, kernel, iterations=2)

                contours, _ = cv2.findContours(
                    mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                motion = False
                annotated = frame.copy()
                for cnt in contours:
                    if cv2.contourArea(cnt) > min_area:
                        motion = True
                        x, y, w, h = cv2.boundingRect(cnt)
                        cv2.rectangle(annotated, (x, y), (x+w, y+h),
                                      (0, 255, 0), 2)
                status = 'MOTION' if motion else 'static'
                color  = (0, 0, 255) if motion else (0, 255, 0)
                cv2.putText(annotated, status, (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
                results.append((idx, motion, annotated))
            prev_gray = gray
        idx += 1

    cap.release()
    return results

results = detect_motion('/tmp/cv_video/sample.mp4', threshold=20, min_area=300)

motion_frames   = [(idx, fr) for idx, mot, fr in results if mot]
static_frames   = [(idx, fr) for idx, mot, fr in results if not mot]
print(f'Total sampled frames: {len(results)}')
print(f'Frames with motion  : {len(motion_frames)}')
print(f'Static frames       : {len(static_frames)}')

# Show a mix of examples
examples = results[:6]
show_frames(
    [fr for _, _, fr in examples],
    [f'Frame {idx} — {"MOTION" if mot else "static"}'
     for idx, mot, _ in examples],
    cols=3, figsize=(15, 5)
)

## Summary

| Topic | Key function / concept |
|---|---|
| Open video | `cv2.VideoCapture(path)` |
| Video properties | `cap.get(cv2.CAP_PROP_FPS/FRAME_COUNT/...)` |
| Read frame | `ret, frame = cap.read()` |
| Write video | `cv2.VideoWriter(path, fourcc, fps, (w,h))` |
| Dense optical flow | `cv2.calcOpticalFlowFarneback()` |
| Sparse optical flow | `cv2.calcOpticalFlowPyrLK()` |
| Feature tracking | `cv2.goodFeaturesToTrack()` + LK flow |
| Motion detection | `cv2.absdiff()` + threshold + contours |

## Practice Exercises

**Exercise 1 — Frame Rate Analyser:**  
Write a function that reads a video and measures the actual playback speed by timing how fast frames are decoded. Compare it to the stored FPS metadata. Compute the total duration from both methods.

**Exercise 2 — Highlight Reel:**  
Using the motion detection results, extract only the frames where motion was detected and stitch them together into a new video using `VideoWriter`. Add a frame number overlay to each.

**Exercise 3 — Optical Flow Magnitude Histogram:**  
For the dense optical flow result, compute the magnitude of the flow vectors using `cv2.cartToPolar`. Plot a histogram of the magnitude values. What does the distribution tell you about the amount of motion in the scene?